# Quantum 2048: planning with shared fate
**Waterloo Qiskit Fall Fest 2026 · Standalone project notebook**

A sliding-number puzzle that makes the joint distribution useful before a move. The contribution is an exact, non-measuring merge forecast tied to a playable ruleset. Quantum 2048 has prior implementations; we do not claim global novelty, speedup, or measured learning gains.

## Run this notebook by itself

Python 3.12 recommended. Install dependencies once in your notebook environment:

```python
%pip install qiskit==2.6.2 numpy==2.5.4
```

Then restart the kernel and **Run All**. The installation command above is documentation, not an automatically executed cell. The full simulator/game implementation and its tests are embedded below. No separate project files, server, browser, IBM credentials, or hardware are needed for these examples.

The implementation is copied verbatim from the shared core used by the playable application. Full application and sources: https://github.com/dsnothing/Quantum-Hackathon

All outputs below are actual local simulator results. Organizer permission for pre-existing work remains unconfirmed.

In [1]:
import sys, importlib.metadata as metadata
import numpy as np
print('Python', sys.version.split()[0], 'Qiskit', metadata.version('qiskit'))

Python 3.12.14 Qiskit 2.6.2


## Embedded shared implementation

This is the complete project engine, included here so the uploaded notebook is self-contained. The examples below exercise this implementation directly.

Source SHA-256: `5a09ab3d63a461b4310c49334ceaff6ee60388a70bde8ec491a631c776eef1f5`

In [2]:
"""Small Qiskit engine and deterministic 2048 reducer shared by UI and notebook."""
from __future__ import annotations
from dataclasses import dataclass
from copy import deepcopy
from itertools import product
import random
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix


class QuantumGroup:
    def __init__(self, linked=False, seed=0):
        self.circuit = QuantumCircuit(2 if linked else 1)
        self.circuit.h(0)
        if linked:
            self.circuit.cx(0, 1)
        self.state = Statevector.from_instruction(self.circuit)
        self.state.seed(seed)

    def probabilities(self):
        return self.state.probabilities_dict()

    def measure(self, qubit):
        result, self.state = self.state.measure([qubit])
        return int(result)

    def marginal(self, qubit):
        return self.state.probabilities([qubit])


def interference():
    """Same engine primitives; preserve or remove phase coherence before H."""
    plus = QuantumGroup().state
    h = QuantumCircuit(1)
    h.h(0)
    coherent = plus.evolve(h).probabilities()
    dephased = DensityMatrix(np.diag(plus.probabilities())).evolve(h).probabilities()
    return {'coherent': coherent.tolist(), 'dephased': dephased.tolist()}


@dataclass
class Tile:
    id: int
    value: int | None = None
    group: int | None = None
    qubit: int = 0


class Game:
    def __init__(self, seed=1, scenario=None):
        self.rng = random.Random(seed)
        self.board = [None] * 16
        self.groups = {}
        self.next_id = 1
        self.next_group = 1
        self.score = self.turn = 0
        self.last = []
        if scenario in ('linked', 'independent', 'tradeoff'):
            if scenario == 'independent':
                self.add_quantum([0]); self.add_quantum([1])
            elif scenario == 'linked':
                self.add_quantum([0, 1])
            else:
                self.add_quantum([0, 6]); self.add_classic(1, 2)
                self.add_classic(4, 8)
            self.add_classic(15 if scenario == 'tradeoff' else 12, 8)
        else:
            self.add_classic(5, 2); self.add_classic(10, 2)
            self.add_quantum([0, 3])

    def add_classic(self, pos, value):
        self.board[pos] = Tile(self.next_id, value)
        self.next_id += 1

    def add_quantum(self, positions):
        gid = self.next_group; self.next_group += 1
        self.groups[gid] = QuantumGroup(len(positions) == 2, self.rng.randrange(2**32))
        for q, pos in enumerate(positions):
            self.board[pos] = Tile(self.next_id, None, gid, q)
            self.next_id += 1

    def support(self, tile):
        if tile.value is not None:
            return {tile.value}
        return {2 << b for b, p in enumerate(self.groups[tile.group].marginal(tile.qubit)) if p > 1e-10}

    def resolve(self, tile, assignment=None):
        if tile.value is not None:
            return False
        if assignment is None:
            bit = self.groups[tile.group].measure(tile.qubit)
            # Conditional state persists, including the distant linked partner.
            for other in self.board:
                if other and other.group == tile.group and other.value is None:
                    p = self.groups[other.group].marginal(other.qubit)
                    if max(p) > 1 - 1e-10:
                        other.value = 2 << int(np.argmax(p))
        else:
            bit = (assignment[tile.group] >> tile.qubit) & 1
            # Branch evaluation emulates conditional Bell collapse without sampling.
            for other in self.board:
                if other and other.group == tile.group and other.value is None:
                    other.value = 2 << ((assignment[other.group] >> other.qubit) & 1)
        tile.value = 2 << bit
        return True

    @staticmethod
    def lines(direction):
        if direction not in ('left', 'right', 'up', 'down'):
            raise ValueError('Unknown direction')
        rows = [[4*r+c for c in range(4)] for r in range(4)]
        cols = [[4*r+c for r in range(4)] for c in range(4)]
        lines = rows if direction in ('left', 'right') else cols
        return [list(reversed(x)) for x in lines] if direction in ('right', 'down') else lines

    def _reduce(self, direction, assignment=None):
        before = [(t.id, t.value) if t else None for t in self.board]
        events, measured = [], False
        for line in self.lines(direction):
            tiles = [self.board[i] for i in line if self.board[i]]
            out, locked = [], set()
            for tile in tiles:
                target = out[-1] if out else None
                if target and target.id not in locked and self.support(target) & self.support(tile):
                    measured = self.resolve(target, assignment) or measured
                    measured = self.resolve(tile, assignment) or measured
                    if target.value == tile.value:
                        events.append((target.id, tile.id, target.value * 2))
                        merged = Tile(self.next_id, target.value * 2)
                        self.next_id += 1
                        self.score += merged.value
                        out[-1] = merged
                        locked.add(merged.id)
                        continue
                out.append(tile)
            for n, i in enumerate(line):
                self.board[i] = out[n] if n < len(out) else None
        after = [(t.id, t.value) if t else None for t in self.board]
        return before != after or measured, events

    def branches(self):
        active = sorted({t.group for t in self.board if t and t.value is None})
        options = [[(int(k, 2), float(p)) for k, p in self.groups[g].probabilities().items() if p > 1e-10] for g in active]
        for combo in product(*options):
            yield dict(zip(active, [x[0] for x in combo])), float(np.prod([x[1] for x in combo]))

    def preview(self, direction):
        events, outcomes = {}, {}
        expected, activity = 0., 0.
        for assignment, p in self.branches():
            copy = deepcopy(self)
            changed, merges = copy._reduce(direction, assignment)
            activity += p * changed
            expected += p * (copy.score - self.score)
            for a, b, v in merges:
                key = (a, b, v)
                events[key] = events.get(key, 0) + p
            key = tuple((t.id, t.value) if t else None for t in copy.board)
            if key not in outcomes:
                outcomes[key] = {'probability': 0., 'board': copy.view()['board']}
            outcomes[key]['probability'] += p
        # Sum both possible output values for a pair's merge probability.
        pairs = {}
        for (a, b, v), p in events.items():
            row = pairs.setdefault((a, b), {'tiles': [a, b], 'probability': 0., 'values': []})
            row['probability'] += p; row['values'].append(v)
        return {'direction': direction, 'merges': list(pairs.values()),
                'expected_score': expected, 'activity': activity,
                'outcomes': sorted(outcomes.values(), key=lambda x: -x['probability'])}

    def move(self, direction, spawn=True):
        changed, events = self._reduce(direction)
        self.last = events
        if changed:
            self.turn += 1
            if spawn: self.spawn()
        return changed

    def spawn(self):
        empty = [i for i, t in enumerate(self.board) if t is None]
        if not empty: return
        unresolved = sum(t is not None and t.value is None for t in self.board)
        if self.turn % 4 == 0 and len(empty) >= 2 and unresolved <= 2:
            self.add_quantum(self.rng.sample(empty, 2))
        elif self.turn % 4 == 2 and unresolved < 4:
            self.add_quantum([self.rng.choice(empty)])
        else:
            self.add_classic(self.rng.choice(empty), 4 if self.rng.random() < .1 else 2)
        active = {t.group for t in self.board if t and t.group is not None}
        self.groups = {g: s for g, s in self.groups.items() if g in active}

    def status(self):
        if any(t and t.value and t.value >= 2048 for t in self.board): return 'won'
        if any(t is None for t in self.board): return 'playing'
        # Any positive-probability active swipe avoids loss, even a resolving collision.
        if any(self.preview(d)['activity'] > 1e-10 for d in ('left','right','up','down')): return 'playing'
        return 'lost'

    def view(self, with_status=False):
        board = []
        for t in self.board:
            if not t: board.append(None); continue
            board.append({'id': t.id, 'value': t.value, 'group': t.group,
                          'linked': t.group is not None and self.groups[t.group].state.num_qubits == 2,
                          'possible': sorted(self.support(t))})
        data = {'board': board, 'score': self.score, 'turn': self.turn, 'last': self.last}
        if with_status: data['status'] = self.status()
        return data


## 1. Rules before circuits

- A 4 × 4 board slides toward the chosen edge. Process rows top to bottom or columns left to right. Within a line, process from the destination edge. Compress, compare consecutive tiles, merge equal values once per swipe. A new merged tile cannot merge again that swipe.
- Fixed tiles hold powers of two. An uncertain tile represents one qubit, with bit 0 mapped to 2 and bit 1 to 4. A linked pair shares a Bell state. Tile IDs survive movement. A merge consumes both IDs and creates one new, fixed tile.
- Measure at a **possible matching contact**: when two candidate supports overlap. Resolve the destination tile then the incoming tile. Measurement updates the joint state and any now-certain partner, wherever it is. The result persists. Disjoint values slide without measurement. A failed uncertain contact still resolves tiles and consumes a turn. Line order matters when partners contact other tiles.
- A swipe is active if positions, values, or measurement state change. A true no-op never spawns or scores. Score is the sum of created merged values. Forecasts exclude the subsequent spawn.
- Start with two ordinary 2s and one linked pair. Each active turn normally spawns a fixed tile in a random empty cell: 90% 2, 10% 4. On turns divisible by four, spawn a Bell pair into two random empty cells if possible and at most four unresolved qubits remain. Otherwise on even turns spawn a single |+⟩ tile if capacity allows. If neither applies, use a fixed tile. Spawn locations and fixed-tile values use classical randomness. Quantum values use Qiskit measurement.
- Reach 2048 to win. A full board loses only when every direction has zero probability of an active move. Restart resets the board and score. The UI stops play after win/loss; the core remains usable for analysis.
- Each quantum group has at most two qubits. Groups are independent of one another. At most four unresolved tile qubits limit forecasts to at most 16 computational-basis branches. Measured qubits stay in their small group until its last tile disappears.

The server holds one local game in memory. Opening a second tab shares that game. No persistence, online multiplayer, accounts, or telemetry.

## 2. The same marginals, different merge forecasts

Independent |+⟩ ⊗ |+⟩ has 00, 01, 10, 11 with probabilities 1/4. Matching outcomes 00 and 11 total **1/2**. A Bell pair (|00⟩ + |11⟩)/√2 has only 00 and 11, so its tiles match with probability **1**. Both cases give each tile a 50/50 marginal.

Qiskit uses little-endian qubit indexing: in a printed two-bit label, the rightmost bit is qubit 0.

In [3]:
single=QuantumGroup()
bell=QuantumGroup(linked=True)
print('Single:', single.probabilities())
print('Bell:', bell.probabilities())
print(bell.circuit.draw(output='text'))
for scenario in ['independent','linked']:
    g=Game(scenario=scenario)
    p=g.preview('left')
    print(scenario, 'match probability:', round(p['merges'][0]['probability'],6),
          'expected score:', round(p['expected_score'],6))
    assert np.isclose(p['merges'][0]['probability'], .5 if scenario=='independent' else 1)
print('Analytical expectations: independent +3, linked +6')

Single: {np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}
Bell: {np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
independent match probability: 0.5 expected score: 3.0
linked match probability: 1.0 expected score: 6.0
Analytical expectations: independent +3, linked +6


## 3. Partial measurement and persistent outcomes

Measuring qubit 0 of a Bell pair selects 0 or 1 and conditions the remaining state. Qubit 1 then has the same bit with probability one. Repeating measurement cannot re-roll a tile. The engine uses Qiskit's returned post-measurement Statevector, rather than inventing a separate partner outcome.

In [4]:
q=QuantumGroup(linked=True,seed=42)
first=q.measure(0)
print('First bit:',first, 'partner probabilities:',q.marginal(1))
print('Repeated:', q.measure(0), 'partner:', q.measure(1))
assert q.state.is_valid()
assert np.isclose(q.marginal(1)[first],1)

First bit: 1 partner probabilities: [0. 1.]
Repeated: 1 partner: 1


## 4. Forecasts and committed moves share one reducer

`Game.preview` reads exact Qiskit basis probabilities, clones the game, and enumerates deterministic branches through `_reduce`. It does not call measurement. Uncontacted tiles remain uncertain in branch boards. `Game.move` uses the same `_reduce`, with actual Qiskit partial measurements at contacts. Since gameplay only measures in the computational basis and introduces no intervening gates, this enumeration reproduces the sequential measurement distribution. Moving a qubit changes its tile position, not its state-vector index.

The browser only displays JSON from this Python core. It does not compute quantum probabilities or randomly choose quantum outcomes.

In [5]:
from copy import deepcopy
g=Game(seed=17,scenario='independent')
snapshot=deepcopy(g)
forecast=g.preview('left')
assert g.view()==snapshot.view()
assert all(np.array_equal(g.groups[k].state.data,snapshot.groups[k].state.data) for k in g.groups)
print('Forecast branches:', [(round(o['probability'],3),
    [t['value'] if t else None for t in o['board'][:4]]) for o in forecast['outcomes']])
g.move('left',spawn=False)
assert any(g.view()['board']==o['board'] for o in forecast['outcomes'])
print('Actual top row:', [t.value if t else None for t in g.board[:4]], 'score:',g.score)
print('Preview left live board and RNG unchanged: passed')

Forecast branches: [(0.25, [4, None, None, None]), (0.25, [2, 4, None, None]), (0.25, [4, 2, None, None]), (0.25, [8, None, None, None])]
Actual top row: [2, 4, None, None] score: 0
Preview left live board and RNG unchanged: passed


## 5. A decision with a cost

The tradeoff fixture puts linked tiles at (row 1, column 1) and (row 2, column 3), a fixed 2 at (1,2), and an 8 at (2,1). Another 8 sits at (4,4). A left swipe contacts the outside 2 and resolves the distant partner. A right swipe also contacts them. An up swipe avoids that contact and keeps the pair uncertain while changing position. A down swipe can also preserve them. Preserving a pair defers information and may require extra moves/spawns. Consuming one partner can earn points immediately but gives up the opportunity to bring the original pair together while uncertain.

This is a concrete decision structure, **not evidence of balanced or enjoyable play**. A pair's two-cell spawn cost prevents it being only an extra free tile. No extra link-score bonus is awarded. Formal balance and user appeal remain unvalidated.

In [6]:
g=Game(scenario='tradeoff')
for d in ['left','up','right','down']:
    p=g.preview(d)
    probe=deepcopy(g);probe.move(d,False)
    unresolved=sum(t is not None and t.value is None for t in probe.board)
    print(d, 'expected score',round(p['expected_score'],3),'unresolved after example move',unresolved)
assert g.preview('left')['expected_score'] > 0
assert g.preview('up')['expected_score'] == 0

left expected score 2.0 unresolved after example move 0
up expected score 0.0 unresolved after example move 2
right expected score 2.0 unresolved after example move 0
down expected score 0.0 unresolved after example move 2


## 6. Coherence: a comparison that matching odds cannot show

A Bell state's Z-basis matching statistics also occur in the classical mixture ½|00⟩⟨00| + ½|11⟩⟨11|. The prepared pure Bell state is entangled, but matching outcomes alone cannot certify that. The game has no demonstrated quantum computational advantage.

Use the project's `QuantumGroup` engine to prepare |+⟩ = H|0⟩. A second H gives |0⟩ by interference. Remove its off-diagonal density-matrix entries first (complete computational-basis dephasing) and the same H gives a 50/50 mixture. Qiskit computes both evolutions. This one-qubit comparison isolates coherence; it is not an entanglement test. We keep it in the notebook to avoid adding an unexplained gate button to the puzzle.

In [7]:
result=interference()
for label,probs in result.items():print(label, 'P(0), P(1):',np.round(probs,6))
np.testing.assert_allclose(result['coherent'],[1,0],atol=1e-12)
np.testing.assert_allclose(result['dephased'],[.5,.5],atol=1e-12)
print('Coherent H-H versus dephased H: verified')

coherent P(0), P(1): [1. 0.]
dephased P(0), P(1): [0.5 0.5]


Coherent H-H versus dephased H: verified


## 7. Focused validation and simulator sampling

Analytical probabilities are exact up to floating-point tolerance. The following sample experiment separately measures actual committed games. The test suite checks merge order, no double merges, partial collapse, repeated measurement, preview purity including RNG, branch membership, linked identities, spawning, restart fixtures, win/loss, interference, and a 160-move invariant run.

### Embedded verification suite

The following tests run without a tests folder or other project files.

In [8]:
import unittest
from copy import deepcopy
import numpy as np



def empty():
    g = Game(); g.board = [None]*16; g.groups = {}; g.score = 0
    return g

class CoreTests(unittest.TestCase):
    def test_no_double_merge_and_directions(self):
        for d, line in [('left',[0,1,2,3]),('right',[3,2,1,0]),('up',[0,4,8,12]),('down',[12,8,4,0])]:
            g=empty()
            for i in line: g.add_classic(i,2)
            g.move(d,False)
            self.assertEqual([g.board[i].value if g.board[i] else None for i in line],[4,4,None,None])
            self.assertEqual(g.score,8)

    def test_three_tiles(self):
        g=empty()
        for i,v in enumerate([2,2,4]):g.add_classic(i,v)
        g.move('left',False)
        self.assertEqual([t.value for t in g.board if t],[4,4])

    def test_analytical_probabilities(self):
        for scenario,p in [('independent',.5),('linked',1.)]:
            g=Game(scenario=scenario)
            self.assertAlmostEqual(g.preview('left')['merges'][0]['probability'],p)
            self.assertAlmostEqual(sum(o['probability'] for o in g.preview('left')['outcomes']),1)

    def test_normalization_partial_and_repeat(self):
        for seed in range(30):
            q=QuantumGroup(True,seed)
            self.assertTrue(q.state.is_valid())
            b=q.measure(0)
            self.assertAlmostEqual(q.marginal(1)[b],1)
            self.assertEqual(q.measure(0),b)
            self.assertEqual(q.measure(1),b)
            self.assertTrue(q.state.is_valid())

    def test_preview_purity_and_outcome_agreement(self):
        for scenario in ['linked','independent','tradeoff']:
            for d in ['left','right','up','down']:
                original=Game(seed=9,scenario=scenario)
                before=deepcopy(original)
                p=original.preview(d)
                self.assertEqual(original.view(),before.view())
                for gid in original.groups:
                    np.testing.assert_array_equal(original.groups[gid].state.data,before.groups[gid].state.data)
                original.move(d,False);before.move(d,False)
                self.assertEqual(original.view(),before.view()) # RNG also unchanged
                result=original.view()['board']
                self.assertTrue(any(result==o['board'] for o in p['outcomes']))

    def test_sampled_forecast(self):
        for scenario,expected in [('independent',.5),('linked',1)]:
            count=0
            for seed in range(400):
                g=Game(seed=seed,scenario=scenario);g.move('left',False)
                count+=bool(g.last)
            self.assertLess(abs(count/400-expected),.08)

    def test_partner_collapse_and_merge_identity(self):
        preserved=Game(scenario='tradeoff');preserved.move('up',False)
        self.assertEqual(sum(t is not None and t.value is None for t in preserved.board),2)
        g=Game(seed=8,scenario='tradeoff')
        gid=g.board[0].group; partner=g.board[6].id
        g.move('left',False)
        pair=[t for t in g.board if t and t.group==gid]
        self.assertTrue(all(t.value in (2,4) for t in pair))
        self.assertIn(partner,[t.id for t in g.board if t])
        g=Game(scenario='linked');ids={g.board[0].id,g.board[1].id};g.move('left',False)
        self.assertFalse(ids & {t.id for t in g.board if t})
        self.assertIsNone(g.board[0].group)

    def test_noop_spawn_restart_win_loss(self):
        g=empty();g.add_classic(0,2)
        self.assertFalse(g.move('left'));self.assertEqual(g.turn,0)
        self.assertTrue(g.move('right'));self.assertEqual(sum(t is not None for t in g.board),2)
        g=empty();g.add_classic(0,1024);g.add_classic(1,1024);g.move('left',False)
        self.assertEqual(g.status(),'won')
        g=empty()
        for i in range(16):g.add_classic(i,2 if (i//4+i%4)%2 else 4)
        self.assertEqual(g.status(),'lost')
        self.assertEqual(Game(seed=3).view(),Game(seed=3).view())

    def test_spawn_cap_and_long_play(self):
        g=Game(seed=21)
        for turn in range(160):
            d=['left','up','right','down'][turn%4]
            g.move(d)
            self.assertLessEqual(sum(t is not None and t.value is None for t in g.board),4)
            self.assertTrue(all(q.state.is_valid() for q in g.groups.values()))
            self.assertEqual(len({t.id for t in g.board if t}),sum(t is not None for t in g.board))

    def test_interference(self):
        result=interference()
        np.testing.assert_allclose(result['coherent'],[1,0],atol=1e-12)
        np.testing.assert_allclose(result['dephased'],[.5,.5],atol=1e-12)




In [9]:
for scenario in ['independent','linked']:
    hits=0
    for seed in range(400):
        trial=Game(seed=seed,scenario=scenario);trial.move('left',False)
        hits+=bool(trial.last)
    print(scenario, f'{hits}/400 swipes merged ({hits/4:.1f}%)')
import unittest, io
suite=unittest.defaultTestLoader.loadTestsFromTestCase(CoreTests)
stream=io.StringIO()
report=unittest.TextTestRunner(stream=stream,verbosity=1).run(suite)
print(stream.getvalue())
assert report.wasSuccessful()

independent 198/400 swipes merged (49.5%)


linked 400/400 swipes merged (100.0%)


..........
----------------------------------------------------------------------
Ran 10 tests in 1.889s

OK



## 8. Contribution, limitations, and usability protocol

**Contribution:** a shared simulator/game reducer with a non-mutating visual forecast, a 50% versus 100% comparison, and a linked-partner positioning cost. An interactive introduction, touch swipe preview, keyboard controls, letter/diamond link markers, and plain-language forecasts aim to make the relationship understandable. These are implemented design choices, not measured usability improvements.

**Limits:** ideal noiseless statevectors; no physical hardware runs; no advantage claim; four unresolved qubits; small hand-designed scenario evaluation rather than balance optimization; preview shows at most four outcome boards in the UI (all outcomes remain in the API); no save game or undo; no measured learning or appeal results. Exact previews condition on the current state and exclude the next spawn. A fixed game seed also fixes classical spawning and Qiskit measurement seeds.

**Short real-user protocol, not yet conducted:** recruit 3–5 volunteers with consent. Ask them to merge ordinary tiles, predict independent odds, predict linked odds, and choose a tradeoff move. Observe whether they find preview/commit, distinguish the link marker without color, and explain distant collapse. Record completion, mistaken predictions, and comments without making learning claims. Ask whether preserving a pair felt costly. Revise instructions or spawning based on actual observations.

**Demo:** load Linked, preview left, show guaranteed merge and 4/8 outcomes, commit. Load Independent and show 50%. Load A harder choice and compare left/up. Show the executed interference comparison. If the browser fails, run sections 2–6 in this notebook.

**Submission status:** public event page requires notebook and explanatory slides, simulator compatible. Presentation required for a chance at a physical certificate. Registration is per person. Public sources do not state cutoff, upload destination, exact slide formats, talk duration, or reuse eligibility. Main event page says November 6, 18:30–21:00; registration says 17:30–20:30 (Waterloo). These are conflicting session times, not verified submission deadlines. Default talk: about four minutes with appendix. Confirm unresolved details before submitting. Nothing has been registered, submitted, or published by this project.

## Sources and acknowledgments

Accessed November 6, 2026 (Waterloo), starting about 03:36 EST.

- [Waterloo official event and artifact requirements](https://uwquantum.com/qiskit-fall-fest)
- [Individual registration and conflicting session schedule](https://uwquantum.com/fall-fest-signup)
- [IBM Qiskit quantum_info reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/quantum_info): Statevector and DensityMatrix. API behavior additionally verified by execution against the pinned installed version.
- [Prior Quantum2048 by nirbhaykwatra](https://github.com/nirbhaykwatra/Quantum2048), MIT-licensed Unity/C# project. Consulted README as prior art. No code or assets copied.
- [Original 2048 by Gabriele Cirulli](https://github.com/gabrielecirulli/2048), MIT-licensed. Credit for the underlying puzzle concept. This implementation uses original Python/HTML/CSS/JavaScript code and no copied assets.

Human team must review scientific claims, supply team identity, rehearse, and confirm organizer eligibility policies. No participant feedback or hardware data is claimed.